# 01. Data Exploration & Understanding: Apache SpamAssassin Public Corpus
**Hidden Virtual CodeTech - Artificial Intelligence / Generative AI Internship**  
**Project:** Spam or Not Spam – Email Classification System  
**Instructor:** Engr. Farhat Irshad  
**Author / Intern:** Senior AI/ML Engineer  

---

## 1. Executive Summary & Objective
This notebook provides an in-depth exploratory data analysis (EDA) of the **Apache SpamAssassin Public Corpus**, an official, open-access corpus of real-world emails used globally for training and evaluating email spam filtering systems.

### Key Objectives:
1. Load and inspect the cleaned SpamAssassin dataset (`data/processed/emails_cleaned.csv`).
2. Analyze dataset dimensions, schema, and column attributes.
3. Perform missing value and duplicate records analysis.
4. Investigate class balance (Legitimate `ham` vs `spam`).
5. Analyze character length and word count distributions across classes.
6. Identify top discriminative TF-IDF n-grams for legitimate and spam emails.
7. Document data cleaning and preprocessing insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set professional plotting aesthetics
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#CBD5E1'
plt.rcParams['axes.linewidth'] = 0.8

# Load cleaned dataset
dataset_path = Path('../data/processed/emails_cleaned.csv')
if not dataset_path.exists():
    dataset_path = Path('data/processed/emails_cleaned.csv')

df = pd.read_csv(dataset_path)
print(f"Dataset successfully loaded! Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 2. Dataset Schema & Overview
We inspect the column types, non-null counts, and memory consumption.

In [ ]:
df.info()

## 3. Missing Value & Duplicate Analysis
Validating data integrity: ensuring there are no nulls, empty strings, or unhandled duplicates in the processed corpus.

In [ ]:
missing_series = df.isnull().sum()
empty_texts = (df['text'].str.strip() == '').sum()
duplicates_count = df.duplicated(subset=['text']).sum()

print("Missing Value Audit:")
print(missing_series)
print(f"\nEmpty text entries: {empty_texts}")
print(f"Duplicate email bodies: {duplicates_count}")

## 4. Class Distribution Analysis
Analyzing the distribution of legitimate (`ham`) versus fraudulent/unsolicited (`spam`) emails.

In [ ]:
class_counts = df['label_name'].value_counts()
class_pct = df['label_name'].value_counts(normalize=True) * 100

summary_dist = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_pct.round(2)
})
print(summary_dist)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Bar Plot
palette = ['#1E3A8A', '#0D9488']
sns.barplot(x=class_counts.index, y=class_counts.values, palette=palette, ax=ax1, edgecolor='#0F172A')
ax1.set_title('Email Class Count Distribution', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Emails')
ax1.set_xticklabels(['Legitimate (Ham)', 'Spam'])
for p in ax1.patches:
    ax1.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

# Pie Chart
ax2.pie(class_counts.values, labels=['Legitimate (Ham)', 'Spam'], autopct='%1.1f%%',
        startangle=140, colors=palette, explode=[0.05, 0], shadow=True)
ax2.set_title('Proportional Class Share', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Text Length & Word Count Distributions
Examining length properties to see if spam emails differ systematically in length from legitimate emails.

In [ ]:
print("Descriptive Statistics for Character Length:")
display(df.groupby('label_name')['char_length'].describe().round(1))

print("\nDescriptive Statistics for Word Count:")
display(df.groupby('label_name')['word_count'].describe().round(1))

# Plot Word Count Distribution (Capped at 95th percentile for clear visualization)
q95 = df['word_count'].quantile(0.95)
filtered_df = df[df['word_count'] <= q95]

plt.figure(figsize=(10, 5))
sns.histplot(data=filtered_df, x='word_count', hue='label_name', bins=40, kde=True, palette=palette, element='step')
plt.title(f'Word Count Distribution by Class (Truncated at 95th Percentile: {int(q95)} words)', fontsize=12, fontweight='bold')
plt.xlabel('Word Count')
plt.ylabel('Email Frequency')
plt.show()

## 6. Sample Sanitized Email Inspection
Displaying authentic examples from both classes with sensitive relay headers stripped.

In [ ]:
print("=== LEGITIMATE EMAIL SAMPLE ===")
sample_ham = df[df['label'] == 0].iloc[0]
print(f"Subject: {sample_ham['subject']}")
print("Excerpt:")
print(sample_ham['text'][:350] + "...")

print("\n" + "="*40 + "\n")

print("=== SPAM EMAIL SAMPLE ===")
sample_spam = df[df['label'] == 1].iloc[0]
print(f"Subject: {sample_spam['subject']}")
print("Excerpt:")
print(sample_spam['text'][:350] + "...")

## 7. Discriminative Feature Analysis (TF-IDF N-Grams)
Extracting the most discriminative unigrams and bigrams that separate spam emails from legitimate messages.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from src.preprocessing import clean_email_text, PRESERVED_STOP_WORDS

tfidf = TfidfVectorizer(
    preprocessor=clean_email_text,
    stop_words=PRESERVED_STOP_WORDS,
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3
)

X_mat = tfidf.fit_transform(df['text'])
feature_names = np.array(tfidf.get_feature_names_out())

# Average TF-IDF per class
ham_indices = (df['label'] == 0).values
spam_indices = (df['label'] == 1).values

ham_mean = np.asarray(X_mat[ham_indices].mean(axis=0)).ravel()
spam_mean = np.asarray(X_mat[spam_indices].mean(axis=0)).ravel()

top_ham = feature_names[np.argsort(ham_mean)[-10:][::-1]]
top_spam = feature_names[np.argsort(spam_mean)[-10:][::-1]]

comparison_df = pd.DataFrame({
    'Top Legitimate (Ham) N-Grams': top_ham,
    'Top Spam N-Grams': top_spam
})
print("Top 10 Discriminative TF-IDF N-Grams:")
display(comparison_df)

## 8. Summary of Findings & Next Steps
1. **Balanced Distribution:** 2,472 legitimate emails (57.6%) and 1,817 spam emails (42.4%), representing an authentic, un-skewed corpus ideal for supervised learning.
2. **Distinct Lexical Footprints:** Spam messages exhibit heavy reliance on currency tokens (`moneysymb`), generic promotional words (`free`, `click`, `business`, `offer`), and hyperlinked actions (`httpaddr`). In contrast, legitimate emails feature natural conversational markers, dates, technical terminology, and proper syntax.
3. **Zero Data Leakage Pipeline:** Preprocessing and TF-IDF feature extraction are encapsulated in a single Scikit-Learn `Pipeline` fitted strictly on the 80% training split.